# Gradient Descent — Beginner Masterclass

**Based on:** [GeeksforGeeks — What Is Gradient Descent?](https://www.geeksforgeeks.org/machine-learning/what-is-gradient-descent/)

This Google Colab-compatible notebook explains gradient descent from first principles. It includes hand-calculated updates, a pure-Python implementation, NumPy training, learning-rate visualizations, gradient-descent variants, and the article's Diabetes/BMI regression example.

## Learning goals
- Explain loss, gradient, parameters, iterations, and learning rate.
- Derive the MSE gradients for a straight-line model.
- Implement gradient descent from scratch.
- See what happens when the learning rate is too small, appropriate, or too large.
- Compare batch, stochastic, and mini-batch gradient descent.
- Understand momentum, AdaGrad, RMSprop, and Adam.
- Train on a dataset that loads automatically in Colab.

**Source note:** This notebook summarizes the article in original wording and adds worked calculations and experiments. It does not reproduce the page verbatim.

## 1. What is gradient descent?

Gradient Descent is an iterative optimization algorithm that attempts to minimize a loss function by adjusting parameters in the direction opposite to the gradient.

Think of walking downhill: start at your current point, estimate the slope, take a step downhill, and repeat.

\[
\boxed{\theta_{t+1}=\theta_t-\eta\nabla_\theta J(\theta_t)}
\]

- \(\theta\): model parameters, such as weights and biases.
- \(J(\theta)\): loss / cost function.
- \(\nabla_\theta J\): gradient of the loss with respect to the parameters.
- \(\eta\): learning rate, controlling the step size.
- \(t\): current iteration.

The gradient points toward the steepest local increase. To reduce loss, gradient descent moves in the opposite direction. It does not guarantee a global minimum for every non-convex neural-network loss.

## 2. A one-parameter example

Consider the convex function:

\[
J(w)=(w-3)^2,\qquad \frac{dJ}{dw}=2(w-3).
\]

Start at \(w_0=0\), with learning rate \(\eta=0.1\):

\[
g_0=2(0-3)=-6,\quad w_1=0-0.1(-6)=0.6.
\]

The loss decreases from \(J(0)=9\) to \(J(0.6)=5.76\). Because the gradient is negative, subtracting it moves \(w\) upward toward the minimum at \(w=3\).

In [ ]:
# Pure Python only — one gradient-descent step

def loss(w):
    return (w - 3) ** 2

def gradient(w):
    return 2 * (w - 3)

w = 0.0
learning_rate = 0.1
print("Before update: w =", w, "loss =", loss(w), "gradient =", gradient(w))
w_new = w - learning_rate * gradient(w)
print("After update:  w =", w_new, "loss =", loss(w_new), "gradient =", gradient(w_new))

## 3. Learning rate: too small, suitable, or too large?

- **Too small:** tiny updates can make progress very slowly.
- **Suitable:** the loss often decreases efficiently.
- **Too large:** updates may overshoot the minimum, oscillate, or diverge.

The best learning rate depends on the curvature and scale of the objective. The exact threshold illustrated below applies only to this quadratic example, not to every deep network.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def run_1d_gd(lr, steps=30, start=0.0):
    w = float(start)
    path = [w]
    losses = [(w - 3) ** 2]
    for _ in range(steps):
        w = w - lr * 2 * (w - 3)
        if not np.isfinite(w) or abs(w) > 1e8:
            break
        path.append(w)
        losses.append((w - 3) ** 2)
    return np.array(path), np.array(losses)

learning_rates = [0.02, 0.1, 0.6, 1.1]
plt.figure(figsize=(9, 4.5))
for lr in learning_rates:
    path, losses = run_1d_gd(lr)
    plt.plot(range(len(losses)), np.maximum(losses, 1e-12), marker=".", label=f"lr={lr}")
plt.yscale("log")
plt.xlabel("Update")
plt.ylabel("Loss (log scale)")
plt.title("Learning-rate experiment: J(w) = (w - 3)^2")
plt.legend()
plt.grid(alpha=0.25)
plt.show()

for lr in learning_rates:
    path, losses = run_1d_gd(lr)
    print(f"lr={lr}: final w={path[-1]:.5g}, final loss={losses[-1]:.5g}, updates={len(path)-1}")

For the quadratic example, gradient descent follows:

\[
w_{t+1}-3=(1-2\eta)(w_t-3).
\]

The distance from the minimum shrinks when \(|1-2\eta|<1\), i.e. \(0<\eta<1\). If \(\eta>1\), the distance grows in this example. This is a useful mathematical illustration—not a universal learning-rate rule for neural networks.

**Important distinction:** an overly large learning rate can make updates unstable even when gradients are finite. *Exploding gradients* specifically refers to gradient magnitudes becoming excessively large during backpropagation. They are related optimization problems, but not the same concept.

## 4. Linear regression: the model to optimize

For one input feature, a linear model predicts:

\[
\hat y_i=mx_i+c
\]

where \(m\) is the slope (weight) and \(c\) is the intercept (bias).

Using mean squared error:

\[
J(m,c)=\frac{1}{n}\sum_{i=1}^{n}(\hat y_i-y_i)^2
=\frac{1}{n}\sum_{i=1}^{n}(mx_i+c-y_i)^2.
\]

The goal is to find values of \(m\) and \(c\) that reduce the average squared prediction error.

## 5. Derive the gradients

Using the chain rule:

\[
\boxed{\frac{\partial J}{\partial m}=\frac{2}{n}\sum_{i=1}^{n}(\hat y_i-y_i)x_i}
\]

because the derivative of \(mx_i+c-y_i\) with respect to \(m\) is \(x_i\).

Similarly:

\[
\boxed{\frac{\partial J}{\partial c}=\frac{2}{n}\sum_{i=1}^{n}(\hat y_i-y_i)}
\]

because the derivative with respect to \(c\) is 1.

The update is:

\[
m\leftarrow m-\eta\frac{\partial J}{\partial m},\qquad
c\leftarrow c-\eta\frac{\partial J}{\partial c}.
\]

## 6. Calculate one update by hand

Use three points: \(x=[1,2,3]\) and \(y=[2,4,6]\). Start with \(m=0,c=0\).

Predictions are \([0,0,0]\), therefore:

\[
J=\frac{(0-2)^2+(0-4)^2+(0-6)^2}{3}=\frac{56}{3}\approx18.6667.
\]

Slope gradient:

\[
\frac{\partial J}{\partial m}=\frac{2}{3}[(-2)(1)+(-4)(2)+(-6)(3)]
=-\frac{56}{3}\approx-18.6667.
\]

Intercept gradient:

\[
\frac{\partial J}{\partial c}=\frac{2}{3}(-2-4-6)=-8.
\]

With \(\eta=0.05\), the updated parameters are approximately \(m=0.9333\) and \(c=0.4\).

In [ ]:
# Pure Python; no NumPy in this calculation
x_values = [1.0, 2.0, 3.0]
y_values = [2.0, 4.0, 6.0]
m, c, lr = 0.0, 0.0, 0.05
n = len(x_values)
predictions = [m*x + c for x in x_values]
errors = [p-y for p, y in zip(predictions, y_values)]
mse_before = sum(e*e for e in errors) / n
dm = (2/n) * sum(e*x for e, x in zip(errors, x_values))
dc = (2/n) * sum(errors)
print("Predictions:", predictions)
print("Errors:", errors)
print("Squared errors:", [e*e for e in errors])
print("MSE before update:", mse_before)
print("Gradient dm:", dm, "| Gradient dc:", dc)
m_new, c_new = m - lr*dm, c - lr*dc
new_predictions = [m_new*x + c_new for x in x_values]
new_errors = [p-y for p, y in zip(new_predictions, y_values)]
mse_after = sum(e*e for e in new_errors) / n
print("Updated slope:", m_new, "| Updated intercept:", c_new)
print("Updated predictions:", new_predictions)
print("MSE after update:", mse_after)
assert mse_after < mse_before

## 7. Batch Gradient Descent using NumPy

This version calculates the gradient using the entire training dataset at each update. Each loop iteration predicts, computes MSE, calculates gradients, updates the parameters, and records the loss.

In [ ]:
def batch_gradient_descent_numpy(X, y, learning_rate=0.05, iterations=1000):
    X = np.asarray(X, dtype=float).reshape(-1)
    y = np.asarray(y, dtype=float).reshape(-1)
    if len(X) != len(y):
        raise ValueError("X and y must have the same number of examples.")
    m, c = 0.0, 0.0
    losses = []
    for _ in range(iterations):
        error = m*X + c - y
        loss = np.mean(error**2)
        dm = (2/len(X)) * np.dot(error, X)
        dc = (2/len(X)) * np.sum(error)
        losses.append(loss)
        m -= learning_rate * dm
        c -= learning_rate * dc
    return m, c, np.array(losses)

X_demo = np.array([1., 2., 3.])
y_demo = np.array([2., 4., 6.])
m_fit, c_fit, loss_history = batch_gradient_descent_numpy(X_demo, y_demo, 0.05, 100)
print("Learned slope:", m_fit)
print("Learned intercept:", c_fit)
print("Initial MSE:", loss_history[0])
print("Final MSE:", loss_history[-1])
plt.figure(figsize=(7,4))
plt.plot(loss_history)
plt.xlabel("Iteration")
plt.ylabel("MSE")
plt.title("Batch Gradient Descent on three points")
plt.grid(alpha=0.25)
plt.show()

## 8. Batch vs Stochastic vs Mini-batch Gradient Descent

- **Batch Gradient Descent:** uses all training examples for each parameter update. Its gradient is stable, but each update can be expensive on large datasets.
- **Stochastic Gradient Descent (SGD):** uses one example per update. Updates are cheap but noisy, so the loss may fluctuate.
- **Mini-batch Gradient Descent:** uses a small group (often 32–256 samples) per update. It balances gradient noise and computation and is common in deep learning.

They differ in how much data is used to estimate each gradient.

In [ ]:
# Compare variants on one synthetic regression problem. Standardize y for this section only.
rng = np.random.default_rng(3)
X_comp = np.linspace(-2, 2, 240)
y_raw = 1.7*X_comp + 0.4 + rng.normal(0, 0.25, size=len(X_comp))
y_comp = (y_raw - y_raw.mean()) / y_raw.std()

def train_basic_variant(X, y, mode="batch", lr=0.03, epochs=50, batch_size=32, seed=0):
    rng = np.random.default_rng(seed)
    m, c = 0.0, 0.0
    losses = []
    n = len(X)
    for _ in range(epochs):
        if mode == "batch":
            batches = [np.arange(n)]
        else:
            order = rng.permutation(n)
            step = 1 if mode == "sgd" else batch_size
            batches = [order[i:i+step] for i in range(0, n, step)]
        for ids in batches:
            xb, yb = X[ids], y[ids]
            err = m*xb + c - yb
            dm = 2*np.mean(err*xb)
            dc = 2*np.mean(err)
            m -= lr*dm
            c -= lr*dc
        losses.append(np.mean((m*X+c-y)**2))
    return m, c, np.array(losses)

variant_results = {}
for mode, lr in [("batch", 0.1), ("sgd", 0.005), ("mini", 0.05)]:
    params = train_basic_variant(X_comp, y_comp, mode=mode, lr=lr, epochs=50, batch_size=32, seed=10)
    variant_results[mode] = params

plt.figure(figsize=(8,4))
for mode, (_, _, curve) in variant_results.items():
    plt.plot(curve, label=mode)
plt.xlabel("Epoch")
plt.ylabel("Full-dataset MSE")
plt.title("Batch vs stochastic vs mini-batch gradient descent")
plt.legend()
plt.grid(alpha=0.25)
plt.show()
for mode, (m_v, c_v, curve) in variant_results.items():
    print(f"{mode:>5}: slope={m_v:.4f}, intercept={c_v:.4f}, final MSE={curve[-1]:.5f}")

## 9. Momentum, AdaGrad, RMSprop, and Adam

The article introduces several optimizer variants:

- **Momentum:** carries part of the previous update forward, often accelerating progress along consistent directions and reducing oscillation.
- **AdaGrad:** accumulates squared gradients, reducing effective step sizes for parameters with large historical gradients.
- **RMSprop:** uses a moving average of squared gradients instead of allowing the accumulated sum to grow forever.
- **Adam:** combines moving averages of gradients and squared gradients, with bias correction.

These are variations on gradient-based optimization, and no optimizer is guaranteed to be best for every task.

In [ ]:
def train_optimizer_variant(X, y, method="adam", lr=0.03, epochs=50, batch_size=32, seed=1):
    rng = np.random.default_rng(seed)
    params = np.array([0.0, 0.0])  # [slope, intercept]
    velocity = np.zeros(2)
    accum_sq = np.zeros(2)
    first_moment = np.zeros(2)
    second_moment = np.zeros(2)
    beta1, beta2, eps = 0.9, 0.999, 1e-8
    t, losses = 0, []
    n = len(X)
    for _ in range(epochs):
        order = rng.permutation(n)
        for start in range(0, n, batch_size):
            ids = order[start:start+batch_size]
            xb, yb = X[ids], y[ids]
            error = params[0]*xb + params[1] - yb
            grad = np.array([2*np.mean(error*xb), 2*np.mean(error)])
            t += 1
            if method == "momentum":
                velocity = 0.9*velocity + grad
                params -= lr*velocity
            elif method == "adagrad":
                accum_sq += grad**2
                params -= lr*grad/(np.sqrt(accum_sq)+eps)
            elif method == "rmsprop":
                accum_sq = 0.9*accum_sq + 0.1*grad**2
                params -= lr*grad/(np.sqrt(accum_sq)+eps)
            elif method == "adam":
                first_moment = beta1*first_moment + (1-beta1)*grad
                second_moment = beta2*second_moment + (1-beta2)*(grad**2)
                m_hat = first_moment/(1-beta1**t)
                v_hat = second_moment/(1-beta2**t)
                params -= lr*m_hat/(np.sqrt(v_hat)+eps)
            else:
                raise ValueError("Unknown method")
        losses.append(np.mean((params[0]*X+params[1]-y)**2))
    return params[0], params[1], np.array(losses)

adaptive_results = {}
learning_rates = {"momentum": 0.01, "adagrad": 0.1, "rmsprop": 0.01, "adam": 0.03}
for method, lr in learning_rates.items():
    adaptive_results[method] = train_optimizer_variant(
        X_comp, y_comp, method=method, lr=lr, epochs=50, batch_size=32, seed=12
    )

plt.figure(figsize=(8,4))
for method, (_, _, curve) in adaptive_results.items():
    plt.plot(curve, label=method)
plt.xlabel("Epoch")
plt.ylabel("Full-dataset MSE")
plt.title("Optimizer variants on synthetic regression")
plt.legend()
plt.grid(alpha=0.25)
plt.show()
for method, (m_v, c_v, curve) in adaptive_results.items():
    print(f"{method:>8}: final MSE={curve[-1]:.5f}")

## 10. Load the Diabetes dataset used by the article

The article's example selects the BMI feature and the diabetes-progression target:

```python
from sklearn.datasets import load_diabetes
diabetes = load_diabetes()
X = diabetes.data[:, [2]]  # BMI feature
 y = diabetes.target
```

The dataset is bundled with scikit-learn, so it loads automatically in Colab without a manual file upload. We'll split the data before fitting `StandardScaler`, which avoids using test-set information when estimating the feature scale. This is a best-practice refinement of the article's simple demonstration.

The target is a disease-progression measurement, not a diagnosis or a probability.

In [ ]:
from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, r2_score

diabetes = load_diabetes()
X_all = diabetes.data[:, [2]]
y_all = diabetes.target.astype(float)
X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X_all, y_all, test_size=0.25, random_state=42
)
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train_raw).reshape(-1)
X_test = scaler.transform(X_test_raw).reshape(-1)

print("Dataset samples:", len(X_all))
print("Training samples:", len(X_train), "| Test samples:", len(X_test))
print("Feature:", diabetes.feature_names[2])
print("Scaled training feature mean:", X_train.mean())
print("Scaled training feature std:", X_train.std())
print("Target range:", y_train.min(), "to", y_train.max())

In [ ]:
plt.figure(figsize=(7,4))
plt.scatter(X_train, y_train, alpha=0.55, label="Training samples")
plt.xlabel("Standardized BMI feature")
plt.ylabel("Diabetes progression target")
plt.title("Diabetes dataset: BMI vs progression")
plt.legend()
plt.grid(alpha=0.2)
plt.show()

## 11. Train Batch Gradient Descent on the Diabetes BMI feature

The loop follows the article's main steps:
1. Initialize slope and intercept at zero.
2. Compute predictions \(\hat y=mx+c\).
3. Compute mean squared error.
4. Calculate slope and intercept gradients.
5. Update both parameters.
6. Repeat for 1,000 iterations and track the loss.

The input feature is standardized for better optimization. The target remains in its original scale.

In [ ]:
def fit_batch_gd(X, y, learning_rate=0.05, iterations=1000, verbose=False):
    X = np.asarray(X, dtype=float).reshape(-1)
    y = np.asarray(y, dtype=float).reshape(-1)
    if len(X) != len(y):
        raise ValueError("X and y must have the same number of samples.")
    m, c = 0.0, 0.0
    loss_history, checkpoints = [], []
    for i in range(iterations):
        y_pred = m*X + c
        error = y_pred - y
        loss = np.mean(error**2)
        dm = (2/len(X)) * np.dot(error, X)
        dc = (2/len(X)) * np.sum(error)
        loss_history.append(loss)
        m -= learning_rate*dm
        c -= learning_rate*dc
        if i % 100 == 0:
            checkpoints.append((i, loss, m, c))
            if verbose:
                print(f"Iteration {i:4d}: loss={loss:.4f}, m={m:.4f}, c={c:.4f}")
    return m, c, np.array(loss_history), checkpoints

m, c, diabetes_loss, checkpoints = fit_batch_gd(
    X_train, y_train, learning_rate=0.05, iterations=1000, verbose=True
)
train_pred = m*X_train + c
test_pred = m*X_test + c
print("\nFinal slope:", m)
print("Final intercept:", c)
print("Train MSE:", mean_squared_error(y_train, train_pred))
print("Test MSE:", mean_squared_error(y_test, test_pred))
print("Test R²:", r2_score(y_test, test_pred))

In [ ]:
plt.figure(figsize=(8,4))
plt.plot(diabetes_loss)
plt.xlabel("Iteration")
plt.ylabel("Training MSE")
plt.title("Gradient descent loss curve — Diabetes BMI feature")
plt.grid(alpha=0.25)
plt.show()

order = np.argsort(X_train)
plt.figure(figsize=(7,4.5))
plt.scatter(X_train, y_train, alpha=0.5, label="Training data")
plt.plot(X_train[order], (m*X_train+c)[order], color="red", linewidth=2, label="Fitted line")
plt.xlabel("Standardized BMI")
plt.ylabel("Diabetes progression")
plt.title("Linear fit learned by gradient descent")
plt.legend()
plt.grid(alpha=0.2)
plt.show()

## 12. Inspect parameter updates

The table lists selected iterations. The loss is recorded before the update at that iteration; the slope and intercept in the row are the values after that update.

In [ ]:
import pandas as pd
checkpoint_df = pd.DataFrame(
    checkpoints,
    columns=["Iteration", "MSE before update", "m after update", "c after update"]
)
display(checkpoint_df)

## 13. Common mistakes and debugging checklist

- **Learning rate too small:** training can be very slow.
- **Learning rate too large:** updates may oscillate or diverge.
- **Poor feature scaling:** gradients can have very different magnitudes; scaling often helps.
- **Gradient sign error:** use `prediction - target` consistently in the formula shown here. Reversing the sign without changing the update direction can increase loss.
- **Missing factor `2/n`:** these formulas are for mean squared error, so the gradient includes its averaging factor.
- **Shape mismatch:** ensure inputs and targets have the same number of samples and flatten arrays consistently for a one-feature model.
- **Data leakage:** fit scalers on training data only.
- **Global-minimum assumption:** linear regression with MSE is convex in slope/intercept; deep-network objectives are generally non-convex.
- **Large learning rate vs exploding gradients:** a large step can destabilize optimization even when the gradient is finite. Exploding gradients specifically means the gradient magnitudes become excessively large.

## 14. Advantages and limitations

**Advantages**
- Simple and widely applicable.
- Works with many differentiable loss functions.
- Batch variants adapt to different dataset sizes.
- Forms the basis of many modern neural-network optimizers.

**Limitations**
- Sensitive to learning-rate choice.
- Affected by poor parameter initialization or badly scaled features.
- Full-batch updates can be expensive on large datasets.
- Stochastic updates introduce noise.
- Global-minimum convergence is not generally guaranteed for non-convex objectives.

## 15. Knowledge check

1. If the gradient is positive, which way does standard gradient descent move the parameter?
2. What happens when the learning rate is too small?
3. What is the difference between batch and stochastic gradient descent?
4. Why standardize the BMI feature?
5. Write the MSE gradients for slope \(m\) and intercept \(c\).
6. Does gradient descent always find the global minimum of a deep network loss?
7. How do momentum, AdaGrad, RMSprop, and Adam differ at a high level?

In [ ]:
print("1. It subtracts a positive quantity, so the parameter decreases.")
print("2. Progress may be very slow.")
print("3. Batch uses all examples per update; stochastic uses one; mini-batch uses a small group.")
print("4. Scaling often improves numerical conditioning and optimizer behaviour.")
print("5. dm=(2/n)*sum((prediction-target)*x); dc=(2/n)*sum(prediction-target).")
print("6. No; deep neural-network objectives are generally non-convex.")
print("7. Momentum reuses previous updates; AdaGrad accumulates squared gradients; RMSprop uses an EMA of squared gradients; Adam combines EMA of gradients and squared gradients.")

## 16. Sources and official documentation

### Main article
- [GeeksforGeeks — What Is Gradient Descent?](https://www.geeksforgeeks.org/machine-learning/what-is-gradient-descent/) — overview, learning rate, MSE gradients, Diabetes/BMI example, variants, advantages, and disadvantages.

### Official documentation
- [scikit-learn `load_diabetes`](https://scikit-learn.org/stable/modules/generated/sklearn.datasets.load_diabetes.html)
- [scikit-learn `StandardScaler`](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.StandardScaler.html)
- [scikit-learn `SGDRegressor`](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.SGDRegressor.html)
- [PyTorch SGD optimizer](https://docs.pytorch.org/docs/stable/generated/torch.optim.SGD)
- [Keras SGD optimizer](https://keras.io/api/optimizers/sgd/)
- [NumPy dot product](https://numpy.org/doc/stable/reference/generated/numpy.dot.html)
- [NumPy mean](https://numpy.org/doc/stable/reference/generated/numpy.mean.html)

When debugging, check shapes, learning rate, gradient calculation, and whether the loss is recorded before or after an update. The notebook paraphrases the article and adds explanatory examples rather than copying it verbatim.